# Qwen-Image-2.1-Turbo 在 Google Colab 上运行

基于 [abenzerps/Qwen-Image-2.1-Turbo-Quantized](https://huggingface.co/abenzerps/Qwen-Image-2.1-Turbo-Quantized)，
在 Colab 免费 T4 (16GB VRAM) 上部署，并提供 **OpenAI 风格 `/v1/images/generations` 接口**（端口 `8010`），
再用 `cloudflared` 暴露公网地址。

## 模型选型（Colab 推荐）

| 量化文件 | 大小 | 建议 |
|---|---|---|
| `qwen-image-2.1-turbo-Q4_K_M.gguf` | 4.05 GB | ✅ Colab 首选，体积与质量平衡 |
| `qwen-image-2.1-turbo-Q4_0.gguf` | 4.05 GB | 显存极紧时的备选 |
| `qwen-image-2.1-turbo-Q5_K_M.gguf` | 4.94 GB | 质量更好，稍慢，T4 16GB 可跑 |
| `qwen-image-2.1-turbo-Q6_K.gguf` | 5.88 GB | 显存吃紧，慎选 |
| `qwen-image-2.1-turbo-Q8_0.gguf` | 7.59 GB | 一般不建议在免费 Colab 上使用 |

必须配套下载（工作流三件套）：
- 文本编码器：`text_encoders/qwen3vl_8b_int8_convrot.safetensors` (9.35 GB，INT8 更省显存)
- VAE：`vae/qwen_image_2.1_vae_bf16.safetensors` (0.68 GB)

合计约 **14 GB** 下载；Colab 免费版磁盘通常充足，下载需 10–30 分钟，视网络而定。

> **注意**：这是文生图（DiT 7B）模型，**不适用 `llama-cpp-python`**（那是文本模型推理库）。
本笔记本采用 ComfyUI + [ComfyUI-GGUF](https://github.com/leejet/ComfyUI-GGUF) 运行 GGUF 权重，
并用 FastAPI 做一层 OpenAI 兼容代理，保持你习惯的 `OpenAI` 客户端调用方式。

运行前：`Runtime → Change runtime type → T4 GPU`，保存后重新运行全部 Cell。


In [ ]:
import sys, os
print('Python:', sys.version)
try:
    import torch
    print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
except Exception as e:
    print('torch 未安装:', e)
!nvidia-smi
!df -h /content | tail -1


In [ ]:
import urllib.request, time
MIRRORS = [
    ("huggingface.co", "https://huggingface.co"),
    ("hf-mirror.com", "https://hf-mirror.com"),
    ("gitcode.com/hf_mirrors", "https://gitcode.com/hf_mirrors"),
]
TEST_PATH = "/abenzerps/Qwen-Image-2.1-Turbo-Quantized/resolve/main/qwen-image-2.1-turbo-Q4_K_M.gguf"

def probe(base):
    req = urllib.request.Request(base + TEST_PATH, method="HEAD")
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=15) as r:
            return round(time.time() - t0, 2), int(r.headers.get('Content-Length', 0)), r.status
    except Exception as e:
        return None, 0, repr(e)[:80]

rows = []
for name, base in MIRRORS:
    dt, size, status = probe(base)
    rows.append((name, base, dt, size, status))
    print(f'{name:26s} {str(dt):>6}s  status={status}  size={size/1e9:.2f} GB')

ok = [r for r in rows if r[2] is not None]
BASE = min(ok, key=lambda r: r[2])[1] if ok else "https://huggingface.co"
print(">>> 选用镜像源:", BASE)


In [ ]:
import os, subprocess, sys
os.chdir('/content')
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
os.chdir('/content/ComfyUI')
!pip install -r requirements.txt -q -i https://pypi.tuna.tsinghua.edu.cn/simple
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    !git clone https://github.com/leejet/ComfyUI-GGUF.git custom_nodes/ComfyUI-GGUF
print('ComfyUI 就绪')


In [ ]:
import os
MODEL_DIR = '/content/ComfyUI/models'
FILES = [
    ('diffusion_models/qwen-image-2.1-turbo-Q4_K_M.gguf', 'qwen-image-2.1-turbo-Q4_K_M.gguf', 4.05e9),
    ('text_encoders/qwen3vl_8b_int8_convrot.safetensors', 'qwen3vl_8b_int8_convrot.safetensors', 9.35e9),
    ('vae/qwen_image_2.1_vae_bf16.safetensors', 'qwen_image_2.1_vae_bf16.safetensors', 0.68e9),
]
for rel, fname, expect in FILES:
    dst = os.path.join(MODEL_DIR, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst) and abs(os.path.getsize(dst) - expect) < expect * 0.05:
        print('已存在，跳过:', rel)
        continue
    url = BASE + '/abenzerps/Qwen-Image-2.1-Turbo-Quantized/resolve/main/' + fname
    print('下载:', url)
    !wget -c -q --show-progress "$url" -O "$dst"
    got = os.path.getsize(dst)
    print('完成:', rel, '%.2f GB' % (got / 1e9))


In [ ]:
import subprocess, sys, time, urllib.request, os
log = open('/content/comfyui.log', 'w')
comfy = subprocess.Popen([sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
                         cwd='/content/ComfyUI', stdout=log, stderr=log)
print('ComfyUI PID:', comfy.pid)
for i in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=3)
        print('ComfyUI 已就绪: http://127.0.0.1:8188')
        break
    except Exception:
        time.sleep(5)
else:
    print('等待超时，查看 /content/comfyui.log')


In [ ]:
# 写 /content/qwen_api.py —— 若 ComfyUI-GGUF 版本节点类名不同，修改下面 UNET/CLIP/VAE 类名即可
from fastapi import FastAPI
from pydantic import BaseModel
import requests, base64, uuid, time, uvicorn

app = FastAPI()
COMFY = 'http://127.0.0.1:8188'
UNET = 'qwen-image-2.1-turbo-Q4_K_M.gguf'
CLIP = 'qwen3vl_8b_int8_convrot.safetensors'
VAE = 'qwen_image_2.1_vae_bf16.safetensors'
UNET_CLASS = 'UNETLoaderGGUF'   # 如报错，按 ComfyUI-GGUF 实际节点类名替换

class ImageGenRequest(BaseModel):
    prompt: str
    model: str = UNET
    width: int = 1024
    height: int = 1024
    steps: int = 8
    seed: int = -1
    cfg: float = 1.0

def build_workflow(req: ImageGenRequest):
    seed = req.seed if req.seed and req.seed >= 0 else uuid.uuid4().int >> 64
    wf = {
        '1': {'inputs': {'unet_name': req.model}, 'class_type': UNET_CLASS},
        '2': {'inputs': {'clip_name': CLIP, 'type': 'qwen_image'}, 'class_type': 'CLIPLoader'},
        '3': {'inputs': {'vae_name': VAE}, 'class_type': 'VAELoader'},
        '4': {'inputs': {'width': req.width, 'height': req.height, 'batch_size': 1}, 'class_type': 'EmptyLatentImage'},
        '5': {'inputs': {'model': ['1', 0], 'positive': ['2', 0], 'negative': ['2', 0], 'latent_image': ['4', 0],
                'seed': int(seed), 'steps': req.steps, 'cfg': req.cfg,
                'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 1.0}, 'class_type': 'KSampler'},
        '6': {'inputs': {'samples': ['5', 0], 'vae': ['3', 0]}, 'class_type': 'VAEDecode'},
        '7': {'inputs': {'images': ['6', 0], 'filename_prefix': 'qwen_colab'}, 'class_type': 'SaveImage'},
    }
    return wf

@app.post('/v1/images/generations')
def generate(req: ImageGenRequest):
    wf = build_workflow(req)
    r = requests.post(COMFY + '/prompt', json={'prompt': wf}, timeout=30)
    r.raise_for_status()
    pid = r.json()['prompt_id']
    for _ in range(120):
        time.sleep(2)
        h = requests.get(COMFY + f'/history/{pid}', timeout=10).json()
        if pid in h and 'outputs' in h[pid]:
            outs = h[pid]['outputs']
            break
    else:
        return {'error': 'generation timeout'}
    images = []
    for node_id, out in outs.items():
        for img in out.get('images', []):
            data = requests.get(COMFY + '/view',
                params={'filename': img['filename'], 'subfolder': img.get('subfolder', ''), 'type': 'output'},
                timeout=60).content
            images.append({'b64_json': base64.b64encode(data).decode('ascii')})
    return {'created': int(time.time()), 'data': images}

if __name__ == '__main__':
    uvicorn.run(app, host='0.0.0.0', port=8010)


In [ ]:
import subprocess, sys, time, urllib.request
api = subprocess.Popen([sys.executable, 'qwen_api.py'], cwd='/content',
                       stdout=open('/content/qwen_api.log', 'w'), stderr=subprocess.STDOUT)
print('FastAPI 代理 PID:', api.pid)
for i in range(40):
    try:
        urllib.request.urlopen('http://127.0.0.1:8010/docs', timeout=3)
        print('FastAPI 已就绪: http://127.0.0.1:8010')
        break
    except Exception:
        time.sleep(2)


In [ ]:
import subprocess, re, time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
tun = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8010'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public = None
for line in tun.stdout:
    print(line, end='')
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public = m.group(0)
        break
print("\n>>> 公网 OpenAI 兼容接口:", public + "/v1/images/generations")


In [ ]:
import requests, base64, time
from IPython.display import Image, display
payload = {
    'prompt': '一只穿宇航服的猫坐在月球表面，背景是地球升起，电影级光照，超细节，4K',
    'width': 1024, 'height': 1024, 'steps': 8, 'seed': 42, 'cfg': 1.0
}
t0 = time.time()
r = requests.post('http://127.0.0.1:8010/v1/images/generations', json=payload, timeout=300)
dt = time.time() - t0
print('HTTP', r.status_code, '| 端到端耗时: %.1f 秒' % dt)
img_b64 = r.json()['data'][0]['b64_json']
open('/content/qwen_image_test.png', 'wb').write(base64.b64decode(img_b64))
display(Image(filename='/content/qwen_image_test.png'))


# 用你自己的 OpenAI 客户端调用

```python
from openai import OpenAI
client = OpenAI(base_url='https://<你的 cloudflared 域名>/v1', api_key='not-needed')
resp = client.images.generate(
    model='qwen-image-2.1-turbo-Q4_K_M.gguf',
    prompt='春日樱花下的江南小镇，水墨风格，8K',
    width=1024, height=1024
)
img_b64 = resp.data[0].b64_json
```

### FAQ
- **跑不起来？** 确认 Colab 运行时选了 **T4 GPU**；本地 CPU 推理 DiT 7B 太慢，不建议。
- **显存不足？** 把 `qwen3vl_8b_int8_convrot.safetensors` 换更小的量化，或把 `width/height` 降到 768。
- **下载慢？** Cell C 会自动探测 huggingface.co / hf-mirror.com / gitcode.com/hf_mirrors 并选最快镜像；pip 走清华源。
- **接口风格：** 保留 OpenAI `images.generate` 兼容；`b64_json` 直接解码为图片。
- **测速指标：** Cell 10 打印单图端到端耗时（含 ComfyUI 排队与去噪），可用于对比不同量化版本。
